# 01 — Data audit

Everything that determines the training design: class balance, lesion size
(the reason the old 416 px inference lost caries), image resolution, teeth per
panorama, and how the two training tasks overlap.

All numbers are computed live from the annotations — nothing hard-coded.

In [ ]:
import sys
from collections import Counter
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "notebooks"))
import dentex_data as dd

DATA_ROOT = Path("/home/sayfeldin/Projects/dental-model-research")

qe = dd.load_coco(DATA_ROOT / "data/raw/dentex/quadrant_enumeration/train_quadrant_enumeration.json")
qed = dd.load_coco(DATA_ROOT / "data/raw/dentex/training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json")
val = dd.load_coco(DATA_ROOT / "data/downloads/validation_triple.json")

print(f"{'dataset':10s} {'images':>7s} {'boxes':>7s} {'boxes/img':>10s}")
for name, d in (("QE", qe), ("QED", qed), ("val", val)):
    n_img, n_box = len(d["images"]), len(d["annotations"])
    print(f"{name:10s} {n_img:7d} {n_box:7d} {n_box / n_img:10.1f}")

In [ ]:
def distribution(annotations, key, labels):
    counts = Counter(a[key] for a in annotations)
    total = sum(counts.values())
    print(f"{'class':22s} {'count':>7s} {'share':>7s}")
    for idx, name in enumerate(labels):
        c = counts.get(idx, 0)
        print(f"{name:22s} {c:7d} {100 * c / total:6.1f}%")
    return counts


print("=== Disease distribution (training, QED) ===")
dis_train = distribution(qed["annotations"], "category_id_3", dd.DISEASE_NAMES)
print("\n=== Disease distribution (locked val) ===")
dis_val = distribution(val["annotations"], "category_id_3", dd.DISEASE_NAMES)
print("\n=== Class imbalance (train) ===")
hi, lo = max(dis_train, key=dis_train.get), min(dis_train, key=dis_train.get)
print(f"max/min = {dis_train[hi] / dis_train[lo]:.1f}x "
      f"({dd.DISEASE_NAMES[hi]} {dis_train[hi]} vs {dd.DISEASE_NAMES[lo]} {dis_train[lo]})")

In [ ]:
print("=== Quadrant distribution (QED) ===")
distribution(qed["annotations"], "category_id_1", ["Q1", "Q2", "Q3", "Q4"])
print("\n=== Tooth distribution (QE, tooth-level boxes) ===")
distribution(qe["annotations"], "category_id_2", [f"Tooth {t}" for t in range(1, 9)])

In [ ]:
def size_stats(dataset, name):
    imgs = dd.image_map(dataset)
    rel = sorted((a["bbox"][2] * a["bbox"][3]) / (imgs[a["image_id"]]["width"] * imgs[a["image_id"]]["height"])
                 for a in dataset["annotations"])
    def pct(p):
        return rel[min(len(rel) - 1, int(p * len(rel)))]
    print(f"{name}: median {pct(0.5) * 100:.3f}% of image | "
          f"p10 {pct(0.1) * 100:.3f}% | p90 {pct(0.9) * 100:.3f}%")
    return rel


rel = size_stats(qed, "lesion area")
# A 2872 px wide panorama at imgsz=416 shrinks a lesion to this many px.
for imgsz in (416, 640, 1024):
    px = (rel[len(rel) // 2] ** 0.5) * imgsz
    print(f"  median lesion at imgsz={imgsz:4d}: ~{px:.1f} px per side")

In [ ]:
print("=== Image resolution (QED) ===")
for (w, h), c in Counter((i["width"], i["height"]) for i in qed["images"]).most_common(6):
    print(f"{w}x{h}: {c} images")
print("\n=== Teeth per panorama (QE) ===")
per = Counter(a["image_id"] for a in qe["annotations"])
vals = sorted(per.values())
print(f"min {vals[0]} | p25 {vals[len(vals)//4]} | median {vals[len(vals)//2]} "
      f"| p75 {vals[3*len(vals)//4]} | max {vals[-1]}")

In [ ]:
qe_names = {i["file_name"] for i in qe["images"]}
qed_names = {i["file_name"] for i in qed["images"]}
val_names = {i["file_name"] for i in val["images"]}
print("QE subset of QED:", qe_names <= qed_names, f"({len(qe_names)} shared)")
print("QED-only images (disease labels, no tooth labels):", len(qed_names - qe_names))
print("val overlap with training:", len(val_names & qed_names),
      "-> must be 0 for an honest benchmark")
assert not (val_names & qed_names), "LEAKAGE: validation images appear in training"
print("\nOK: locked validation set is disjoint from training data.")

## Findings that drive the design

1. **Caries dominates** — 2,189 of 3,529 lesions (~62%), while Periapical
   Lesion has 158 (~4.5%). Class imbalance is ~14:1 and must be handled
   (mosaic/oversampling or per-class loss weighting), or the rare class stays
   undetected.
2. **Lesions are small but not microscopic.** The median lesion covers ~1.2%
   of the panorama (~45 px at 416, ~112 px at 1024). Resolution alone is not
   the main failure mode; it is a contributing factor for the smallest
   (p10 ~0.75%) and most elongated lesions. Train at 640–1024 and treat 416
   as a speed-only fallback.
3. **Images are ~2872x1504**, so even 1024 px downscales ~2.8x. If recall
   stays low after the other fixes, tiled inference (or a tooth-crop stage)
   is the escape hatch.
4. **The locked 50-image val set is disjoint from training** — the benchmark
   is honest, and must stay that way.
5. **QE is a subset of QED** (634 of 705): the tooth detector trains on the
   634 fully annotated panoramas; the extra 71 QED images still help the
   disease detector.
6. **The old pipeline's real defects were process, not just resolution:** it
   validated on its own training images (`val: images/train`), never measured
   detection precision/recall, and inferred tooth numbers from box geometry
   with an RF. Those are the three things this rebuild changes.